In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

first_drop  = pd.read_csv("../../model_v1/first_drop.csv",  index_col=0)
later_drops = pd.read_csv("../../model_v1/later_drops.csv", index_col=0)

In [ ]:
N_SIM  = 1000
races  = sorted(later_drops['Race'].unique())
rng    = np.random.default_rng(seed=42)

rows = []

for race in races:
    race_drops = later_drops[later_drops['Race'] == race].sort_values('Batch_ID')

    n_d       = race_drops['New_Total'].values.astype(int)
    p_d       = race_drops['New_Dem_Prop'].values
    batch_ids = race_drops['Batch_ID'].values

    # Reconstruct integer Dem vote counts from proportions
    dem_d   = np.round(p_d * n_d).astype(int)
    D_total = int(dem_d.sum())
    N_total = int(n_d.sum())

    cum_N        = np.cumsum(n_d)
    actual_prop  = np.cumsum(dem_d) / cum_N   # point estimate at each step

    # ------------------------------------------------------------------
    # Permutation CI via sequential hypergeometric sampling.
    #
    # Equivalent to: shuffle all N_total votes and split into drops of
    # sizes n_d[0], n_d[1], ..., tracking the cumulative Dem proportion.
    # Using the hypergeometric is O(n_drops) per sim instead of O(N_total).
    # ------------------------------------------------------------------
    D_rem       = np.full(N_SIM, D_total)          # (N_SIM,) Dem left in pool
    N_rem       = np.full(N_SIM, N_total)          # (N_SIM,) total left in pool
    cum_dem_sim = np.zeros(N_SIM, dtype=np.int64)  # (N_SIM,) cumulative Dem drawn

    lower = np.empty(len(n_d))
    upper = np.empty(len(n_d))

    for j, n_j in enumerate(n_d):
        draws        = rng.hypergeometric(D_rem, N_rem - D_rem, n_j)
        cum_dem_sim += draws
        D_rem       -= draws
        N_rem       -= n_j                          # same n_j subtracted for every sim

        sim_props = cum_dem_sim / cum_N[j]
        lower[j]  = np.percentile(sim_props, 2.5)
        upper[j]  = np.percentile(sim_props, 97.5)

    for j, bid in enumerate(batch_ids):
        rows.append({
            'Race':           race,
            'Batch_ID':       bid,
            'point_estimate': actual_prop[j],
            'lower_95':       lower[j],
            'upper_95':       upper[j],
        })

baseline_df = pd.DataFrame(rows)

# Attach first-drop baseline for reference line
baseline_dict = dict(zip(first_drop['Race'], first_drop['Dem_Prop_Before_11_08']))
baseline_df['first_drop'] = baseline_df['Race'].map(baseline_dict)

# ------------------------------------------------------------------
# Plot — same FacetGrid layout as model_v1
# ------------------------------------------------------------------
def plot_baseline(data, **kwargs):
    ax = plt.gca()
    ax.fill_between(data['Batch_ID'], data['lower_95'], data['upper_95'],
                    alpha=0.25, color='gray')
    ax.plot(data['Batch_ID'], data['point_estimate'],
            color='gray', marker='o', markersize=4)
    ax.axhline(data['first_drop'].iloc[0], color='black',
               linestyle='--', alpha=0.6)

sns.set_theme(style="whitegrid")
g = sns.FacetGrid(baseline_df, col='Race', col_wrap=5,
                  sharey=False, height=3, aspect=1.2)
g.map_dataframe(plot_baseline)
g.set_axis_labels("Batch ID", "Cumulative Dem Prop")
g.set_titles(col_template="{col_name}")

handles = [
    plt.Line2D([0], [0], color='gray',  marker='o', markersize=4, label='Cumulative avg'),
    plt.Line2D([0], [0], color='black', linestyle='--', alpha=0.6, label='First drop'),
]
g.axes[0].legend(handles=handles, fontsize=7)
plt.tight_layout()
plt.show()